# Edge AI Deployment Pipeline

This notebook loads the trained model from the original notebook and performs:

1. **ONNX conversion** of the scikit-learn Logistic Regression pipeline
2. **Validation**: numerical comparison between sklearn and ONNX Runtime
3. **Latency benchmark**: sklearn vs ONNX Runtime
4. **Enhanced Raspberry Pi inference script** with P99 latency, power monitoring (vcgencmd), and SQLite logging
5. **Dashboard launch**: the Gradio monitoring dashboard

---
**Prerequisites**: Run the original `Edge_AI_Menstrual_Health_final.ipynb` first to generate the saved model artifacts.

## 1. Setup and imports

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

# Add src to path so we can import the package
sys.path.insert(0, str(Path.cwd().parent))

from edge_ai.models.onnx_utils import convert_to_onnx, validate_onnx, benchmark_latency
from edge_ai.monitoring.metrics import setup_database, measure_all, log_inference, measure_latency
from edge_ai.xai.explainer import Explainer

print("All imports successful.")

## 2. Configure paths

Update `MODEL_DIR` to point to your exported model files from the original notebook.

In [ ]:
# --------------------------------------------------
# Point this to where the original notebook exported files
# --------------------------------------------------
MODEL_DIR = Path("/content/drive/MyDrive/EdgeAI_Project/models/raspberrypi_test")

# If running locally, uncomment and update:
# MODEL_DIR = Path.home() / "edge_ai" / "models"

MODEL_PATH = MODEL_DIR / "final_edge_ai_symptom_risk_pipeline.joblib"
FEATURE_PATH = MODEL_DIR / "final_edge_ai_input_feature_names.joblib"
THRESHOLD_PATH = MODEL_DIR / "final_edge_ai_threshold.joblib"
SAMPLE_INPUT_PATH = MODEL_DIR / "edge_sample_input.csv"
ONNX_PATH = MODEL_DIR / "final_edge_ai_symptom_risk_pipeline.onnx"

print(f"Model path: {MODEL_PATH}")
print(f"ONNX path: {ONNX_PATH}")
print(f"All source files exist: {all(p.exists() for p in [MODEL_PATH, FEATURE_PATH, THRESHOLD_PATH, SAMPLE_INPUT_PATH])}")

## 3. Load trained model and metadata

In [ ]:
pipeline = joblib.load(MODEL_PATH)
feature_names = joblib.load(FEATURE_PATH)
threshold = joblib.load(THRESHOLD_PATH)
sample_df = pd.read_csv(SAMPLE_INPUT_PATH)

# Ensure sample has expected features
sample_df = sample_df[feature_names]

print(f"Pipeline type: {type(pipeline).__name__}")
print(f"Features: {len(feature_names)}")
print(f"Threshold: {threshold}")
print(f"Sample shape: {sample_df.shape}")
print(f"Sample columns: {list(sample_df.columns)}")

## 4. Verify sklearn prediction works

In [ ]:
probs = pipeline.predict_proba(sample_df)[:, 1]
preds = (probs >= threshold).astype(int)

print(f"Example probabilities (first 5): {probs[:5].round(4)}")
print(f"Example predictions (first 5): {preds[:5]}")
print(f"Positive rate: {preds.mean():.3f}")
print("Sklearn pipeline works correctly.")

## 5. Convert to ONNX

In [ ]:
print("Converting pipeline to ONNX...")
convert_to_onnx(pipeline, ONNX_PATH, sample_df)

onnx_size_kb = ONNX_PATH.stat().st_size / 1024
joblib_size_kb = MODEL_PATH.stat().st_size / 1024

print(f"ONNX model saved to: {ONNX_PATH}")
print(f"ONNX model size: {onnx_size_kb:.2f} KB")
print(f"Joblib model size: {joblib_size_kb:.2f} KB")
print(f"Size ratio (ONNX/joblib): {onnx_size_kb / joblib_size_kb:.2f}x")

## 6. Validate ONNX correctness

Check that ONNX Runtime produces the same probabilities as sklearn within tolerance.

In [ ]:
validation = validate_onnx(ONNX_PATH, pipeline, sample_df)

print(f"Max absolute difference: {validation['max_abs_difference']:.6e}")
print(f"Mean absolute difference: {validation['mean_abs_difference']:.6e}")
print(f"Validation PASSED: {validation['passed']}")
print(f"Sklearn output shape: {validation['sklearn_shape']}")
print(f"ONNX output shape: {validation['onnx_shape']}")

if not validation['passed']:
    print("WARNING: ONNX output differs from sklearn! Check the conversion.")

## 7. Latency benchmark (sklearn vs ONNX Runtime)

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(str(ONNX_PATH))

print("Running 1000-iteration latency benchmark...")
bench = benchmark_latency(pipeline, session, sample_df, n_runs=1000)

results_df = pd.DataFrame({
    "Metric": ["Mean", "Median", "Std", "P95", "P99", "Min", "Max"],
    "sklearn (ms)": [
        bench["sklearn"]["mean_ms"], bench["sklearn"]["median_ms"],
        bench["sklearn"]["std_ms"], bench["sklearn"]["p95_ms"],
        bench["sklearn"]["p99_ms"], bench["sklearn"]["min_ms"],
        bench["sklearn"]["max_ms"],
    ],
    "ONNX (ms)": [
        bench["onnx"]["mean_ms"], bench["onnx"]["median_ms"],
        bench["onnx"]["std_ms"], bench["onnx"]["p95_ms"],
        bench["onnx"]["p99_ms"], bench["onnx"]["min_ms"],
        bench["onnx"]["max_ms"],
    ],
})

display(results_df.round(4))

speedup = bench["sklearn"]["mean_ms"] / bench["onnx"]["mean_ms"]
print(f"ONNX speedup over sklearn (mean): {speedup:.2f}x")
print("Benchmark complete.")

## 8. Generate enhanced Raspberry Pi inference script

This script includes:
- P50/P95/P99 latency measurement
- Power monitoring via vcgencmd (voltage, throttling, temperature)
- RAM and CPU monitoring
- SQLite logging for dashboard consumption
- Works with both joblib and ONNX Runtime backends

In [ ]:
# Copy the source run_edge_inference.py to the model directory
import shutil
from pathlib import Path

repo_root = Path.cwd().parent if (Path.cwd() / "src").exists() else Path.cwd()
src = repo_root / "run_edge_inference.py"
dst = MODEL_DIR / "run_edge_inference.py"

if src.exists():
    shutil.copy2(src, dst)
    print(f"Script copied: {src.name} -> {dst}")
    print(f"Size: {dst.stat().st_size / 1024:.1f} KB")
else:
    print(f"WARNING: Source script not found at {src}")
    print("Make sure run_edge_inference.py exists in the repo root.")


## 9. Update requirements.txt for Raspberry Pi

In [ ]:
requirements = """pandas
numpy
scikit-learn
joblib
psutil
onnxruntime
gradio
plotly
"""

req_path = MODEL_DIR / "requirements.txt"
with open(req_path, "w") as f:
    f.write(requirements.strip() + "\n")

print(f"Requirements saved to: {req_path}")
print()
print("Files ready for Raspberry Pi:")
for f in sorted(MODEL_DIR.iterdir()):
    print(f"  - {f.name}")

## 10. Test XAI explanation on sample data

In [ ]:
explainer = Explainer(
    model=pipeline,
    feature_names=feature_names,
    background_df=sample_df,
    threshold=threshold,
)

# Explain a single sample
result = explainer.explain(sample_df.iloc[[0]])

print(f"Probability: {result['probability']:.4f}")
print(f"Risk Level: {result['risk_level']}")
print(f"Explanation method: {result['explanation']['method']}")
print()

top = result['explanation'].get('top_features', [])
if top:
    print("Top contributing factors:")
    for feat in top:
        val = feat.get('shap_value', feat.get('coefficient', 0))
        print(f"  {feat['feature']}: {val:.4f} ({feat['impact_direction']})")

print()
print("Planning Card:")
print(Explainer.make_planning_card(result['explanation']))

## 11. Initialize monitoring database

The database is created at `~/.edge_ai_monitoring.db` and stores:
- inference_logs: per-inference metrics (latency, RAM, CPU, power, prediction)
- system_snapshots: periodic system health metrics

In [ ]:
from edge_ai.monitoring.metrics import setup_database, log_system_snapshot

db_path = setup_database()
print(f"Database initialized at: {db_path}")

# Take an initial snapshot
snap_id = log_system_snapshot()
print(f"Initial system snapshot logged (id={snap_id})")

## 12. Launch the Gradio dashboard (optional)

Run this cell to start the monitoring dashboard locally.
Access it at `http://localhost:7860`.

To launch from the command line:
```bash
pip install -e .
edge-dashboard
```

In [ ]:
# Copy model artifacts to ~/edge_ai_models/ for the dashboard
import shutil

DASHBOARD_MODEL_DIR = Path.home() / "edge_ai_models"
DASHBOARD_MODEL_DIR.mkdir(parents=True, exist_ok=True)

for src in [MODEL_PATH, FEATURE_PATH, THRESHOLD_PATH, SAMPLE_INPUT_PATH]:
    dst = DASHBOARD_MODEL_DIR / src.name
    shutil.copy2(src, dst)
    print(f"Copied {src.name} -> {dst}")

print(f"\nDashboard can now use: --model-dir {DASHBOARD_MODEL_DIR}")

In [ ]:
# Option 1: Launch dashboard inline (if running notebook locally)
# from edge_ai.dashboard.app import run
# run(db_path=db_path, share=False, port=7860, model_manager=explainer)

# Option 2: Launch dashboard from command line (recommended for Pi)
# python -m edge_ai.dashboard.app --model-dir ~/edge_ai_models

---

## Summary

| Task | Status |
|---|---|
| ONNX conversion | Done — model saved alongside joblib |
| ONNX validation | Verified — probabilities match sklearn |
| Latency benchmark | sklearn vs ONNX Runtime compared |
| Enhanced inference script | Generated with P99, power, SQLite |
| XAI explanation | Works with SHAP or coefficient fallback |
| Monitoring database | Initialized at `~/.edge_ai_monitoring.db` |
| Gradio dashboard | Ready to launch |

### Next steps on Raspberry Pi

1. Copy `models/` directory to the Pi
2. Run `bash setup_pi.sh` to install dependencies
3. Run `python run_edge_inference.py` to test inference
4. Launch dashboard: `python -m edge_ai.dashboard.app --model-dir ~/edge_ai_models`
5. Check `raspberry_pi_edge_results.csv` for summary metrics